In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### installing the package

In [2]:
!pip install yahooquery

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 65.2 MB/s eta 0:00:00
  Attempting uninstall: lxml
    Found existing installation: lxml 5.3.0
    Uninstalling lxml-5.3.0:
      Successfully uninstalled lxml-5.3.0


# importing the libraries

In [3]:
import pandas as pd
import yfinance as yf
import datetime
from datetime import date, timedelta

In [4]:
end_date = date.today()
start_date = end_date - timedelta(days=730)  # 2 years = 730 days

from yahooquery import Ticker
btc = Ticker('BTC-USD')

# Fetch historical data
data = btc.history(start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval='1d')

# Reset index if needed and ensure correct column naming
data = data.reset_index()

# Filter data only for BTC-USD if 'symbol' column exists, if other BTC-USD, it removes them
if 'symbol' in data.columns:
    data = data[data['symbol'] == 'BTC-USD']

/usr/local/lib/python3.10/dist-packages/yahooquery/utils/__init__.py:1470: FutureWarning: 'S' is deprecated and will be removed in a future version. Please use 's' instead of 'S'.
  has_live_indice = index_utc[-1] >= last_trade - pd.Timedelta(2, "S")


In [5]:
data.head()

,symbol,date,open,high,low,close,volume,adjclose
0,BTC-USD,2023-03-01,23150.929688,23880.632812,23088.626953,23646.550781,24662841200,23646.550781
1,BTC-USD,2023-03-02,23647.019531,23739.138672,23245.021484,23475.466797,20386398516,23475.466797
2,BTC-USD,2023-03-03,23476.632812,23479.347656,22213.238281,22362.679688,26062404610,22362.679688
3,BTC-USD,2023-03-04,22362.923828,22405.177734,22198.980469,22353.349609,11166012913,22353.349609
4,BTC-USD,2023-03-05,22354.144531,22613.685547,22307.142578,22435.513672,13317001733,22435.513672


In [6]:
data.tail()

,symbol,date,open,high,low,close,volume,adjclose
726,BTC-USD,2025-02-24,96277.960938,96503.453125,91371.742188,91418.171875,44046480529,91418.171875
727,BTC-USD,2025-02-25,91437.117188,92511.078125,86008.234375,88736.171875,92139104128,88736.171875
728,BTC-USD,2025-02-26,88638.890625,89286.250000,82131.898438,84347.023438,64597492134,84347.023438
729,BTC-USD,2025-02-27,84076.859375,87000.781250,83144.960938,84704.226562,52659591954,84704.226562
730,BTC-USD,2025-02-28 13:27:00+00:00,84671.164062,84770.367188,78411.039062,80685.062500,82173378560,80685.062500


In [7]:
data.shape

(731, 8)

In [8]:
data.columns

Index(['symbol', 'date', 'open', 'high', 'low', 'close', 'volume', 'adjclose'], dtype='object')

In [9]:
import plotly.graph_objects as go
figure = go.Figure(data=[go.Candlestick(x=data['date'],
                                        open=data['open'], 
                                        high=data['high'],
                                        low=data['low'], 
                                        close=data['close'])])
figure.update_layout(title="Bitcoin Price Analysis", 
                     xaxis_rangeslider_visible=False)
figure.show()

In [10]:
#finding the correlation by selecting only numbers
correlation = data.select_dtypes(include=['number']).corr()
print(correlation["close"].sort_values(ascending=False))

close       1.000000
adjclose    1.000000
high        0.999257
low         0.998982
open        0.998025
volume      0.665623
Name: close, dtype: float64


In [11]:
!pip install autots

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 971.7/971.7 kB 21.5 MB/s eta 0:00:00


In [12]:
from autots import AutoTS

# Ensure 'date' is in datetime format
data['date'] = pd.to_datetime(data['date'], utc=True)

# Convert all timestamps to timezone-naive (removes timezone info)
data['date'] = data['date'].dt.tz_localize(None)

model = AutoTS(forecast_length=30, frequency='infer', ensemble='simple')
model = model.fit(data, date_col='date', value_col='close', id_col=None)
prediction = model.predict()
forecast = prediction.forecast
print(forecast)

Using 1 cpus for n_jobs.
Data frequency is: D, used frequency is: D
Model Number: 1 with model AverageValueNaive in generation 0 of 25
Model Number: 2 with model AverageValueNaive in generation 0 of 25
Model Number: 3 with model AverageValueNaive in generation 0 of 25
Model Number: 4 with model DatepartRegression in generation 0 of 25
Model Number: 5 with model DatepartRegression in generation 0 of 25
Model Number: 6 with model DatepartRegression in generation 0 of 25


/usr/local/lib/python3.10/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning:

Liblinear failed to converge, increase the number of iterations.

/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:541: ConvergenceWarning:

lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html



Model Number: 7 with model DatepartRegression in generation 0 of 25


/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - loss: 0.4242
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4160
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4155
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4123
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4146
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.4104
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.4076
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4043
Epoch 9/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.4039
Epoch 10/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3987
Epoch 11/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3967
Epoch 12/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3954
Epoch 13/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3968
Epoch 14/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3965
Epoch 15/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3913
Epoch 16/50
22/22 ━

...

113 - Ensemble with avg smape 11.27: 
114 - Ensemble with avg smape 11.27: 
115 - Ensemble with avg smape 11.57: 
116 - Ensemble with avg smape 12.9: 
117 - Ensemble with avg smape 23.31: 
118 - Ensemble with avg smape 22.52: 
119 - Ensemble with avg smape 16.73: 
120 - Ensemble with avg smape 18.44: 
121 - Ensemble with avg smape 11.27: 
122 - Ensemble with avg smape 21.43: 
123 - Ensemble with avg smape 14.21: 


/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning:

invalid value encountered in less_equal

/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning:

invalid value encountered in less_equal

/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning:

invalid value encountered in less_equal

/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning:

invalid value encountered in less_equal

/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning:

invalid value encountered in less_equal

/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning:

invalid value encountered in less_equal

/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning:

invalid value encountered in less_equal

/usr/local/li

124 - Ensemble with avg smape 11.34: 
125 - Ensemble with avg smape 14.89: 
126 - Ensemble with avg smape 14.4: 


/usr/local/lib/python3.10/dist-packages/autots/evaluator/auto_model.py:3041: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/usr/local/lib/python3.10/dist-packages/autots/evaluator/auto_model.py:3041: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_coordinate_descent.py:631: ConvergenceWarning:

Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.556e+00, tole

                            close
2025-03-01 13:27:00  80685.062500
2025-03-02 13:27:00  87288.966029
2025-03-03 13:27:00  86295.944866
2025-03-04 13:27:00  85297.552671
2025-03-05 13:27:00  84255.529397
2025-03-06 13:27:00  83228.347613
2025-03-07 13:27:00  82186.836816
2025-03-08 13:27:00  81134.009210
2025-03-09 13:27:00  80127.052656
2025-03-10 13:27:00  79145.468168
2025-03-11 13:27:00  78197.904139
2025-03-12 13:27:00  77249.162161
2025-03-13 13:27:00  76351.013505
2025-03-14 13:27:00  75473.958779
2025-03-15 13:27:00  74617.505328
2025-03-16 13:27:00  73830.018863
2025-03-17 13:27:00  73092.657383
2025-03-18 13:27:00  72402.918069
2025-03-19 13:27:00  71726.736643
2025-03-20 13:27:00  71107.878158
2025-03-21 13:27:00  70515.834673
2025-03-22 13:27:00  69946.542212
2025-03-23 13:27:00  69440.065026
2025-03-24 13:27:00  68980.287085
2025-03-25 13:27:00  68555.121389
2025-03-26 13:27:00  68133.642877
2025-03-27 13:27:00  67753.985724
2025-03-28 13:27:00  67387.305395
2025-03-29 13: